# 03 — Baseline Backtests
Short puts and iron condors from YAML configs, unconditioned vs
signal-conditioned (VIX rank / RSI). Every run lands in the results store.

In [ ]:
import sys
sys.path.insert(0, "../src")
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.width", 160)

In [ ]:
from lab.backtest import StrategyConfig, run_backtest
from lab.experiments import save_run
from lab.report import compare_runs, regime_breakdown

START, END = "2016-01-01", "2023-12-31"   # widen to 2010 for the full study

base = StrategyConfig.from_yaml("../configs/short_put_45dte.yaml").replace(start=START, end=END)
base_run = run_backtest(base)
save_run(base_run, tag="baseline")
base_run.equity_curve.plot(title=f"{base.name} — equity")
{k: round(v, 3) for k, v in base_run.metrics.items()}

## Conditioned entries: only sell puts when vol is elevated / oversold

In [ ]:
variants = {
    "vix_rank>0.5":            "vix_rank > 0.5",
    "vix_rank>0.8":            "vix_rank > 0.8",
    "rsi<40 & vix_rank>0.5":   "vix_rank > 0.5 and rsi14 < 40",
}
runs = {"unconditioned": base_run}
for label, expr in variants.items():
    cfg = base.replace(name=f"short_put|{label}", entry_filter=expr)
    runs[label] = run_backtest(cfg)
    save_run(runs[label], tag="baseline")

ax = None
for label, r in runs.items():
    ax = r.equity_curve.plot(ax=ax, label=label)
ax.legend(); ax.set_title("Signal-conditioned short puts");

In [ ]:
compare_runs(tag="baseline").round(3)

## Iron condor baseline

In [ ]:
ic = StrategyConfig.from_yaml("../configs/iron_condor_45dte.yaml").replace(start=START, end=END)
ic_run = run_backtest(ic)
save_run(ic_run, tag="baseline")
ic_run.equity_curve.plot(title=f"{ic.name} — equity")
{k: round(v, 3) for k, v in ic_run.metrics.items()}

## Where does the P&L come from? Performance by VIX-rank regime

In [ ]:
regime_breakdown(base_run.trade_log, feature="vix_rank", bins=4).round(3)